This is designed to extract data from previous experiment runs for future reference.

In [ ]:
import h5py
from datetime import datetime
from sipyco import pyon
import sys
# add repository path
sys.path.append("../")

from repository.imaging.processor import AbsImage, AbsImageSettings

class HDF5DataExtractor:
    def __init__(self, filename):
        self.filename = filename
        with h5py.File(filename, "r") as f:
            self._h5_data = self._hdf5_to_dict(f)
            self.expid = pyon.decode(f["expid"][()])
            self.devarg_overrides = self.expid.get("devarg_overrides")
            self.log_level = self.expid.get("log_level")
            self.exp_file = self.expid.get("file")
            self.name = self.expid.get("class_name")
            self.args = self.expid.get("arguments")
            self.repo_rev = self.expid.get("repo_rev")

        self.rid = self._h5_data["rid"]
        self.artiq_version = self._h5_data["artiq_version"]
        self.start_time = datetime.fromtimestamp(self._h5_data["start_time"])
        self.datasets = self._h5_data["datasets"]

        try:
            self._run_absimg()
        except Exception as e:
            print(f"Failed to process AbsImage data. Check that the required datasets are present and correctly formatted. Error: {e}")
            self.absimg = None

    def _run_absimg(self):
        self.absimg = AbsImage(
            data=self.datasets["Images.absorption.TOF"],
            ref=self.datasets["Images.absorption.REF"],
            bg=self.datasets["Images.absorption.BG"],
            settings = AbsImageSettings.from_dataset(self.datasets["Images.absorption.settings"])
        )


    def _hdf5_to_dict(self, obj):
        out = {}
        if isinstance(obj, h5py.Group):
            for key, item in obj.items():
                out[key] = self._hdf5_to_dict(item)
        elif isinstance(obj, h5py.Dataset):
            data = obj[()]
            # Convert bytes to string if possible
            if isinstance(data, bytes):
                try:
                    data = data.decode()
                except Exception:
                    pass
            out = data
        return out

    def get_args(self, print_non_defaults=False):
        # Extract the run's parameters from the HDF5 file which don't match their defaults.
        vals = {}

        for k, v in self.args.items():
            if k == "ndscan_params":
                params = pyon.decode(self.args["ndscan_params"])
                overrides = params["overrides"]
                for k, v in overrides.items():
                    spec = params["schemata"][k]["spec"]
                    val = v[0]["value"]
                    default = type(val)(params["schemata"][k]["default"])
                    if print_non_defaults:
                        if val != default:
                            print(f"{k}: {val} (default: {default})")
                    if isinstance(val, (int, float)):
                        vals[k] = [val,[val / spec.get("scale", 1), spec.get("unit", "")]]
                    else:
                        vals[k] = [val, ""]
            else:
                vals[k] = [v, ""]
        return vals
    
    def __str__(self):
        # Print the experiment's parameters and metadata in a human-readable format
        return f"{self.name} [{self.filename}]\n\trid {self.rid}\n\t{self.start_time.strftime('%H:%M:%S %Y-%m-%d')}\n\tArtiq {self.artiq_version}"



In [ ]:
from ndscan import results


# filename = (
#     "/storage/Artiq_result_files/results/2026-06-12/12/000024341-AbsorptionImageExpFrag.h5"
# )

filename = (
    "/home/ae19663/artiq/results/2026-06-17/17/000024676-AbsorptionImageExpFrag.h5"
)
# filename = ("/home/ae19663/artiq/results/2026-06-15/17/000024581-analysis_result.h5")
run = HDF5DataExtractor(filename)
keys = run.datasets.keys()
_ = run.absimg.plot()